# Aqua Epoch — Phase 3: reproducible model release and API
Use a **complete project folder**, not this notebook alone. On Colab put it in `MyDrive/Aqua_Epoch` and approve Drive mounting. On local Jupyter start from the project root or notebooks directory. No GPU or LLM API is required.
Colab defaults to rebuilding the evaluation/release in its own Python environment. A saved scikit-learn pickle from another runtime is **not assumed compatible**. Local mode reuses `artifacts/phase3` unless you set REBUILD_RELEASE=True.
This is a competition demo, not certified water safety. Quick classification always abstains from a reliable safety assessment; pH estimates have limited reliability.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
from datetime import datetime, timezone
IN_COLAB = importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Aqua_Epoch')
else:
    ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ml' / 'promote.py').exists()), None)
if ROOT is None or not (ROOT / 'ml' / 'promote.py').is_file():
    raise FileNotFoundError('Set ROOT to the complete project folder containing ml/promote.py')
sys.path.insert(0, str(ROOT))
DATASET = ROOT / 'Aqua_Epoch_BIS_Water_Quality_Dataset.xls - Aqua_Epoch_BIS_Water_Quality_Dataset.csv'
REBUILD_RELEASE = os.getenv('AQUA_REBUILD_RELEASE', '1' if IN_COLAB else '0') == '1'
INSTALL = os.getenv('AQUA_SKIP_INSTALL', '0') != '1'
RELEASE = ROOT / 'artifacts' / 'phase3'
print('Root:', ROOT, '\nRebuild in this runtime:', REBUILD_RELEASE)

In [ ]:
if INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements-dev.txt')], cwd=ROOT, check=True)
# If previously loaded packages were upgraded, restart runtime and rerun.
# For saved releases, install their requirements-serving.txt and match Python major/minor.

In [ ]:
if REBUILD_RELEASE:
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    SOURCE = ROOT / 'reports' / ('phase3_' + stamp + '_evaluation')
    RELEASE = ROOT / 'artifacts' / ('phase3_' + stamp)
    subprocess.run([sys.executable, '-m', 'ml.evaluate', '--dataset', str(DATASET), '--output', str(SOURCE), '--folds', '3', '--trees', '80', '--permutation-repeats', '3'], cwd=ROOT, check=True)
    subprocess.run([sys.executable, '-m', 'ml.promote', '--source', str(SOURCE), '--dataset', str(DATASET), '--output', str(RELEASE)], cwd=ROOT, check=True)
else:
    assert (RELEASE / 'manifest.json').is_file(), 'No release. Set REBUILD_RELEASE=True or run the documented promotion command.'
print('Release:', RELEASE)

## Test the real app, without requiring a public tunnel
FastAPI TestClient runs the same routes locally in this notebook. These are in-process API calls, not a hosted deployment. Full classification needs 18 keys; standalone pH needs 17 (never pH/Potability). Every required key must be present; null means genuinely unmeasured and triggers warnings. Quick mode needs Turbidity, Hardness, TDS.

In [ ]:
from fastapi.testclient import TestClient
from backend.app import create_app
app = create_app(artifacts_dir=str(RELEASE))
client = TestClient(app)
health = client.get('/health').json()
assert health['ready'], f'Release unavailable: {health}. Match recorded runtime or rebuild here.'
catalog = client.get('/models').json()
print(health)
# Illustrative synthetic input assembled from training medians, NOT a real measured sample.
full_input = {name: bounds['median'] for name, bounds in catalog['models']['classification_full']['reference_ranges'].items()}
full = client.post('/predict/full', json=full_input)
assert full.status_code == 200, full.text
print(full.json())

In [ ]:
quick_input = {name: full_input[name] for name in ['Turbidity', 'Hardness', 'TDS']}
quick = client.post('/classify/quick', json=quick_input).json()
assert quick['assessment'] == 'insufficient_evidence'
print('Quick screening:', quick)
ph_input = {name: value for name, value in full_input.items() if name != 'pH'}
ph = client.post('/ph/full', json=ph_input)
assert ph.status_code == 200, ph.text
assert ph.json()['reliability'] == 'limited'
print('pH estimate (no measured pH supplied):', ph.json())

In [ ]:
import pandas as pd
import numpy as np
# Only trusted locally promoted bundles; hashes/runtime were checked by the loader.
model = app.state.artifact_store.load('classification_full')
features = model.metadata['feature_order']
p_safe = float(model.pipeline.predict_proba(pd.DataFrame([[full_input[f] for f in features]], columns=features))[0, 1])
assert np.isclose(full.json()['classification']['probability_safe'], p_safe, atol=1e-12, rtol=0)
assert full.json()['classification']['class_id'] == int(p_safe >= model.metadata['threshold'])
missing = client.post('/classify/full', json={**full_input, 'pH': None})
assert missing.status_code == 200 and missing.json()['assessment'] == 'insufficient_evidence'
assert client.post('/predict/quick', json={**quick_input, 'TDS': True}).status_code == 422
assert client.post('/ph/full', json=full_input).status_code == 422
print('Named-frame parity, stored threshold, missing-input warning and boundary checks passed.')

## Optional live loopback verification
Set RUN_HTTP=True below to start a temporary localhost-only uvicorn server, send real HTTP requests, compare all six routes with direct pipelines, and save timings. It stops automatically. This does not publish your app or require a tunnel. Deployment is a later phase.

In [ ]:
RUN_HTTP = False
if RUN_HTTP:
    from backend.verify_release import verify_release
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    evidence = verify_release(RELEASE, DATASET, ROOT / 'reports' / 'phase3' / ('http_' + stamp + '.json'), samples=30)
    print(evidence['routes'])
client.close()

## Interpretation
Always display `assessment` and warnings alongside `model_label`. The label is only the thresholded dataset prediction; it must not override `insufficient_evidence`. `probability_safe` is P(class 1), not label confidence. No probability is claimed calibrated and no pH confidence interval is fabricated. Input ranges are engineering/training checks, not BIS limits.
Local notebook execution is tested; Google-hosted Colab execution and public deployment still require end-to-end verification in the deployment phase.